In [ ]:
!pip install scikit-learn==1.5.2 # need to downgrade due to incompatibility with xgboost
!pip install xgboost
!pip install pandas cleanlab

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Assume your text file is named 'input.txt' and is in the same directory
# If not, provide the correct path to the file
file_path = '/content/DT.csv'
lines = []

try:
    with open(file_path, 'r') as f:
        for line in f:
            lines.append(line.strip())  # Remove leading/trailing whitespace and newlines
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
    lines = [] # Initialize as empty list if file not found

if lines:
    # Initialize TF-IDF Vectorizer
    # You can adjust parameters like stop_words, max_features, etc.
    vectorizer = TfidfVectorizer()

    # Fit and transform the lines
    tfidf_matrix = vectorizer.fit_transform(lines)

    # You can now work with the tfidf_matrix
    # For example, to see the feature names (words)
    # print("Features (words):", vectorizer.get_feature_names_out())

    # To see the TF-IDF matrix (sparse format)
    # print("TF-IDF Matrix:")
    # print(tfidf_matrix)

    # To convert the sparse matrix to a dense array (less memory efficient for large datasets)
    # print("TF-IDF Matrix (dense):")
    # print(tfidf_matrix.todense())

    # Display the results (optional, depending on what you want to do next)
    print("TF-IDF vectorization complete.")
else:
    print("No lines to process.")

TF-IDF vectorization complete.


In [ ]:
from sklearn.cluster import KMeans

# You need to choose the number of clusters (k)
# For example, let's choose 3 clusters. You might need to experiment with this value.
n_clusters = 5

# Initialize and fit KMeans
# Using n_init='auto' to suppress the warning about the default number of initializations
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto')
kmeans.fit(tfidf_matrix)

# Get the cluster labels for each line
cluster_labels = kmeans.labels_

# Display the cluster labels
print("Cluster labels for each line:")
print(cluster_labels)
print(cluster_labels.shape)


Cluster labels for each line:
[2 2 1 ... 2 3 2]
(38001,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from collections import defaultdict

# Assuming you have `tfidf_matrix` and `cluster_labels` from previous steps

# Iterate through each cluster
for cluster_id in np.unique(cluster_labels):
    print(f"\nAnalyzing duplicates in Cluster {cluster_id}...")

    # Get the indices of data points in the current cluster
    cluster_indices = np.where(cluster_labels == cluster_id)[0]

    if len(cluster_indices) > 1:
        # Extract the TF-IDF vectors for the current cluster
        cluster_tfidf_matrix = tfidf_matrix[cluster_indices]

        # Calculate the cosine similarity matrix for the current cluster
        cluster_cosine_sim_matrix = cosine_similarity(cluster_tfidf_matrix)





Analyzing duplicates in Cluster 0...

Analyzing duplicates in Cluster 1...

Analyzing duplicates in Cluster 2...

Analyzing duplicates in Cluster 3...

Analyzing duplicates in Cluster 4...


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from collections import defaultdict

# Assuming you have `tfidf_matrix` and `cluster_labels` from previous steps

total_duplicates_across_all_clusters = 0

# Iterate through each cluster
for cluster_id in np.unique(cluster_labels):
    print(f"\nAnalyzing duplicates in Cluster {cluster_id}...")

    # Get the indices of data points in the current cluster
    cluster_indices = np.where(cluster_labels == cluster_id)[0]

    if len(cluster_indices) > 1:
        # Extract the TF-IDF vectors for the current cluster
        cluster_tfidf_matrix = tfidf_matrix[cluster_indices]

        # Calculate the cosine similarity matrix for the current cluster
        cluster_cosine_sim_matrix = cosine_similarity(cluster_tfidf_matrix)

        # Dictionary to store groups of duplicate indices (original indices)
        duplicate_groups = {}
        threshold = 1.0

        # Iterate through the upper triangle of the within-cluster cosine similarity matrix
        for i in range(cluster_cosine_sim_matrix.shape[0]):
            for j in range(i + 1, cluster_cosine_sim_matrix.shape[1]):
                if cluster_cosine_sim_matrix[i, j] == threshold:
                    # Get the original indices of the duplicate pair
                    original_index_i = cluster_indices[i]
                    original_index_j = cluster_indices[j]

                    # Check if either index is already in a duplicate group
                    found_group = None
                    for group_id, group in duplicate_groups.items():
                        if original_index_i in group or original_index_j in group:
                            found_group = group_id
                            break

                    if found_group is not None:
                        # Add both indices to the existing group
                        duplicate_groups[found_group].add(original_index_i)
                        duplicate_groups[found_group].add(original_index_j)
                    else:
                        # Create a new group with both indices
                        # Use a unique identifier for the group (e.g., the first index)
                        duplicate_groups[original_index_i] = {original_index_i, original_index_j}

        # Now, process the duplicate_groups to count the number of duplicates in each group
        print(f"Duplicate groups (similarity 1.0) in Cluster {cluster_id}:")
        cluster_total_duplicates = 0
        if duplicate_groups:
            for group_id, group in duplicate_groups.items():
                # The number of duplicates is the size of the group minus 1 (the original)
                num_duplicates = len(group) - 1
                # Only report groups with actual duplicates (size > 1)
                if num_duplicates > 0:
                    print(f"  Group starting with index {group_id}: {num_duplicates} duplicate(s) - Indices: {sorted(list(group))}")
                    cluster_total_duplicates += num_duplicates
            print(f"\nTotal number of unique duplicate groups (similarity 1.0) in Cluster {cluster_id}: {len(duplicate_groups)}")
            print(f"Total duplicates in Cluster {cluster_id}: {cluster_total_duplicates}")
            total_duplicates_across_all_clusters += cluster_total_duplicates

        else:
            print("  No exact duplicate groups found.")

    else:
        print(f"Cluster {cluster_id} has only one data point. No duplicates possible.")

print(f"\nTotal number of duplicates across all unique duplicate groups in all clusters: {total_duplicates_across_all_clusters}")


Analyzing duplicates in Cluster 0...
Duplicate groups (similarity 1.0) in Cluster 0:
  Group starting with index 8: 18 duplicate(s) - Indices: [8, 1718, 3370, 4633, 7592, 7780, 11408, 12780, 14252, 17027, 17624, 23323, 24503, 25236, 26392, 30121, 31192, 35010, 35740]
  Group starting with index 31: 18 duplicate(s) - Indices: [31, 301, 7159, 7242, 11563, 15174, 15822, 16480, 17059, 19101, 22787, 24957, 28637, 33469, 33838, 34329, 34687, 36753, 37518]
  Group starting with index 36: 18 duplicate(s) - Indices: [36, 39, 10083, 10131, 11170, 17413, 19221, 19611, 20852, 23260, 24420, 26277, 26716, 27030, 29410, 31144, 31553, 35201, 35960]
  Group starting with index 47: 18 duplicate(s) - Indices: [47, 309, 4180, 4420, 5035, 6309, 6569, 10226, 19379, 22063, 24772, 25288, 25308, 26126, 26212, 29313, 31987, 34933, 36351]
  Group starting with index 61: 18 duplicate(s) - Indices: [61, 3033, 5245, 6054, 8431, 9360, 10754, 12305, 12873, 15857, 16202, 17486, 20262, 23833, 26121, 26726, 30054, 3423

In [ ]:
import pandas as pd

# Get all duplicate indices from all clusters
duplicate_indices_to_remove = set()

for cluster_id in np.unique(cluster_labels):
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    if len(cluster_indices) > 1:
        cluster_tfidf_matrix = tfidf_matrix[cluster_indices]
        cluster_cosine_sim_matrix = cosine_similarity(cluster_tfidf_matrix)
        duplicate_groups = {}
        threshold = 1.0

        for i in range(cluster_cosine_sim_matrix.shape[0]):
            for j in range(i + 1, cluster_cosine_sim_matrix.shape[1]):
                if cluster_cosine_sim_matrix[i, j] == threshold:
                    original_index_i = cluster_indices[i]
                    original_index_j = cluster_indices[j]

                    found_group = None
                    for group_id, group in duplicate_groups.items():
                        if original_index_i in group or original_index_j in group:
                            found_group = group_id
                            break

                    if found_group is not None:
                        duplicate_groups[found_group].add(original_index_i)
                        duplicate_groups[found_group].add(original_index_j)
                    else:
                        duplicate_groups[original_index_i] = {original_index_i, original_index_j}

        for group in duplicate_groups.values():
            # Keep only one (e.g., min index), mark others as duplicates
            min_index = min(group)
            duplicates_to_remove = group - {min_index}
            duplicate_indices_to_remove.update(duplicates_to_remove)



In [ ]:
# Create cleaned list of lines (excluding duplicates)
cleaned_lines = [line for i, line in enumerate(lines) if i not in duplicate_indices_to_remove]

# Save to DataFrame
df_cleaned_lines = pd.DataFrame({'text': cleaned_lines})

# Optionally save to CSV
# df_cleaned.to_csv('cleaned_output.csv', index=False)

print(f"Original lines: {len(lines)}")
print(f"Removed duplicates: {len(duplicate_indices_to_remove)}")
print(f"Cleaned lines: {len(df_cleaned_lines)}")

Original lines: 38001
Removed duplicates: 21114
Cleaned lines: 16887


In [ ]:
df_cleaned_lines.head()

,text
0,"age,height_cm,weight_kg,heart_rate,blood_press..."
1,"24,194,103,58.7,109.1,6.8,8.58,4.81,yes,M,1"
2,"69,163,35,64.0,125.2,10.1,3.85,2.8,0,M,0"
3,"79,199,98,82.7,124.3,9.2,0.47,2.45,0,M,0"
4,"55,173,107,73.0,124.3,7.2,0.81,3.05,yes,M,0"


In [ ]:

# Load the original CSV file into a DataFrame
df_original = pd.read_csv(file_path)

# The lines list includes the header row as the first line (lines[0]),
# duplicate_indices_to_remove will include indices that are off by -1 if applied directly to df_original.
# Adjust duplicate indices by subtracting 1 (to match df_original)
adjusted_duplicate_indices = {i - 1 for i in duplicate_indices_to_remove if i > 0}

# Drop duplicates from the original DataFrame
df_cleaned = df_original.drop(index=adjusted_duplicate_indices).reset_index(drop=True)

print(f"Original rows: {len(df_original)}")
print(f"Removed duplicates: {len(adjusted_duplicate_indices)}")
print(f"Cleaned rows: {len(df_cleaned)}")

Original rows: 38000
Removed duplicates: 21114
Cleaned rows: 16886


In [ ]:
df_cleaned.head()

,age,height_cm,weight_kg,heart_rate,blood_pressure,sleep_hours,nutrition_quality,activity_index,smokes,gender,is_fit
0,24,194,103,58.7,109.1,6.8,8.58,4.81,yes,M,1
1,69,163,35,64.0,125.2,10.1,3.85,2.80,0,M,0
2,79,199,98,82.7,124.3,9.2,0.47,2.45,0,M,0
3,55,173,107,73.0,124.3,7.2,0.81,3.05,yes,M,0
4,58,181,106,59.2,109.8,8.8,7.59,1.42,no,F,0


In [ ]:
# Separate features and target
X_cleaned = df_cleaned.drop("is_fit", axis=1)
y_cleaned = df_cleaned["is_fit"].astype(int)

# One-hot encode categorical vars
categorical_features = ["gender", "smokes"]
X_cleaned = pd.get_dummies(X_cleaned, columns=categorical_features)


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from cleanlab.classification import CleanLearning

# Define base model
clf = HistGradientBoostingClassifier()

# Wrap with Cleanlab
cl = CleanLearning(clf)

# Find noisy labels
label_issues = cl.find_label_issues(X_cleaned, y_cleaned)
print(label_issues.head())


   is_label_issue  label_quality  given_label  predicted_label
0           False       0.989781            1                1
1           False       0.358342            0                1
2           False       0.900484            0                0
3           False       0.992350            0                0
4           False       0.969574            0                0


In [ ]:
# Get indices of flagged noisy labels
noisy_idx = label_issues[label_issues["is_label_issue"]].index

print(f"Number of noisy labels detected: {len(noisy_idx)}")

# Remove noisy examples
X_final = X_cleaned.drop(index=noisy_idx)
y_final = y_cleaned.drop(index=noisy_idx)

# Retrain model on cleaned labels
cl.fit(X_final, y_final)


Number of noisy labels detected: 56


CleanLearning(clf=HistGradientBoostingClassifier(),
              find_label_issues_kwargs={'confident_joint': array([[9889,   24],
       [  32, 6941]]),
                                        'min_examples_per_class': 10})

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Baseline (before cleaning)
baseline_scores = cross_val_score(clf, X_cleaned, y_cleaned, cv=5, scoring="accuracy")

# After cleaning
cleaned_scores = cross_val_score(clf, X_final, y_final, cv=5, scoring="accuracy")

print(f"Baseline mean accuracy: {np.mean(baseline_scores):.3f}")
print(f"After cleaning mean accuracy: {np.mean(cleaned_scores):.3f}")


Baseline mean accuracy: 0.982
After cleaning mean accuracy: 0.986


Analysis:

Removing noisy labels slightly improved accuracy, showing that label quality was already quite high, but Cleanlab still found subtle errors that were worth correcting.


